In [73]:
# Agregar más puntos y mostrar colores de tráfico en en mapa

In [74]:
#!pip install osmnx folium geopy pandas --upgrade

In [75]:
# Importar las librerías necesarias
import osmnx as ox
import networkx as nx
import folium
import pandas as pd
import heapq
from geopy.distance import geodesic
import time
import math
import os
import pickle
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import requests  #  llamar a la API de TomTom

In [76]:
# ===================== FUNCIONES TOMTOM =====================
import os
import requests
import folium

TOMTOM_API_KEY = os.getenv("TOMTOM_API_KEY2")

if not TOMTOM_API_KEY:
    raise ValueError("ERROR: La variable de entorno TOMTOM_API_KEY no está configurada.")

# -------- Routing (ya lo tenías) --------
def tomtom_route_eta(lat_o, lon_o, lat_d, lon_d):
    """
    Llama a la Routing API de TomTom y devuelve:
      - distance_m: distancia total en metros
      - travel_s: tiempo total de viaje (segundos, con condiciones actuales)
      - delay_s: retraso por tráfico (segundos, si TomTom lo reporta)
    """
    url = f"https://api.tomtom.com/routing/1/calculateRoute/{lat_o},{lon_o}:{lat_d},{lon_d}/json"

    params = {
        "key": TOMTOM_API_KEY,
        "routeType": "fastest",
        "traffic": "true",   # considerar tráfico
        "departAt": "now"    # condiciones actuales
    }

    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()

    summary = data["routes"][0]["summary"]

    distance_m = summary.get("lengthInMeters")
    travel_s   = summary.get("travelTimeInSeconds")
    delay_s    = summary.get("trafficDelayInSeconds", 0)

    return distance_m, travel_s, delay_s


def tomtom_from_graph_nodes(G, start_node, end_node):
    """
    Wrapper que toma nodos del grafo OSMnx y llama a tomtom_route_eta.
    """
    lat_o, lon_o = G.nodes[start_node]["y"], G.nodes[start_node]["x"]
    lat_d, lon_d = G.nodes[end_node]["y"], G.nodes[end_node]["x"]

    return tomtom_route_eta(lat_o, lon_o, lat_d, lon_d)


# -------- Traffic Flow (NUEVO: para colorear tráfico) --------
def tomtom_flow_segment(lat, lon, zoom=10):
    """
    Llama a la Traffic Flow 'Flow Segment Data' API de TomTom, estilo 'absolute'.
    Devuelve el objeto flowSegmentData completo.
    Docs:
    https://developer.tomtom.com/traffic-api/documentation/tomtom-maps/traffic-flow/flow-segment-data
    """
    url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/{zoom}/json"
    params = {
        "key": TOMTOM_API_KEY,
        "point": f"{lat},{lon}",
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    return data["flowSegmentData"]


def traffic_color_from_ratio(ratio, road_closure=False):
    """
    Convierte el ratio currentSpeed/freeFlowSpeed en un color tipo tráfico.
    """
    if road_closure:
        return "black"
    if ratio is None:
        return "gray"
    if ratio >= 0.9:
        return "green"
    elif ratio >= 0.7:
        return "yellow"
    elif ratio >= 0.4:
        return "orange"
    else:
        return "red"


def build_traffic_map_from_segments(segments, center_lat, center_lon, save_path=None):
    """
    Construye un mapa Folium con segmentos coloreados según el tráfico.
    segments: lista de dicts con claves:
      - 'coords': lista [(lat, lon), ...]
      - 'color': string ('green', 'yellow', 'orange', 'red', 'black', 'gray')
    """
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

    for seg in segments:
        coords = seg.get("coords", [])
        color = seg.get("color", "blue")
        if coords:
            folium.PolyLine(coords, color=color, weight=6).add_to(m)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        m.save(save_path)

    return m

# ============================================================


In [77]:
def mapa_ruta_con_trafico(G, astar_edges, traffic_segments,
                          origen="origen", destino="destino",
                          output_dir="Resultados/Mapas_Combinados"):
    import os
    import folium

    if not astar_edges:
        print("[INFO] No hay aristas en la ruta, no se genera mapa.")
        return None

    start_node = astar_edges[0][0]
    center_lat = G.nodes[start_node]["y"]
    center_lon = G.nodes[start_node]["x"]

    m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

    # Ruta en azul
    coords_route = []
    for (u, v) in astar_edges:
        coords_route.append((G.nodes[u]["y"], G.nodes[u]["x"]))
    last_v = astar_edges[-1][1]
    coords_route.append((G.nodes[last_v]["y"], G.nodes[last_v]["x"]))

    folium.PolyLine(coords_route, color="blue", weight=4, opacity=0.8,
                    tooltip="Ruta base / con tráfico").add_to(m)

    # Segmentos de tráfico coloreados
    if traffic_segments:
        for seg in traffic_segments:
            coords = seg.get("coords", [])
            color = seg.get("color", "gray")
            if coords:
                folium.PolyLine(
                    coords,
                    color=color,
                    weight=1,       # más delgado
                    opacity=0.35,   # más transparente
                    tooltip=(
                        f"Vel. actual: {seg.get('currentSpeed')} km/h | "
                        f"Libre: {seg.get('freeFlowSpeed')} km/h | "
                        f"ratio: {round(seg.get('ratio', 0), 2) if seg.get('ratio') else 'N/A'}"
                    )
                ).add_to(m)



    os.makedirs(output_dir, exist_ok=True)

    def slugify(name):
        return str(name).replace(" ", "_").replace("/", "-")

    filename = f"mapa_ruta_con_trafico_{slugify(origen)}_{slugify(destino)}.html"
    save_path = os.path.join(output_dir, filename)

    m.save(save_path)
    print(f"[OK] Mapa combinado guardado en: {save_path}")

    return m


In [78]:
def clean_graph(G, network_type):
    max_speed = {'drive': 40, 'walk': 5}
    for u, v, k in G.edges(keys=True):
        length = G.edges[u, v, k]["length"]
        max_speed_value = max_speed[network_type]

        # Penalización por tipo de calle
        penalty = 1.5 if "highway" in G.edges[u, v, k] and G.edges[u, v, k]["highway"] in ["residential", "tertiary"] else 1.0

        G.edges[u, v, k]["maxspeed"] = max_speed_value
        G.edges[u, v, k]["weight"] = (length / max_speed_value) * penalty
    return G

In [79]:
# Función para inicializar estilos de aristas
def initialize_edge_styles(G):
    for edge in G.edges:
        G.edges[edge]["color"] = "#d36206"
        G.edges[edge]["alpha"] = 0.2
        G.edges[edge]["linewidth"] = 0.5

In [80]:
# Solicitar las coordenadas de inicio y destino
def get_coordinates():
    while True:
        try:
            coords = input("Ingrese las coordenadas (latitud, longitud): ").strip()
            lat, lon = map(float, coords.split(","))
            return (lat, lon)
        except ValueError:
            print("Entrada inválida. Por favor, ingrese las coordenadas en el formato 'lat,lon'.")

In [81]:
# Solicitar el tipo de transporte
def get_transport_type():
    while True:
        print("Ingrese el tipo de viaje que realizará:")
        print("  1. Auto")
        print("  2. Caminando")
        try:
            option = int(input("Seleccione una opción (1 o 2): ").strip())
            if option in [1, 2]:
                return 'drive' if option == 1 else 'walk'
            else:
                print("Opción inválida. Por favor seleccione 1 o 2.")
        except ValueError:
            print("Entrada inválida. Por favor ingrese un número (1 o 2).")

In [82]:
# Algoritmo de Dijkstra
def dijkstra(G, orig, dest):
    for node in G.nodes:
        G.nodes[node]["visited"] = False
        G.nodes[node]["distance"] = float("inf")
        G.nodes[node]["previous"] = None
    G.nodes[orig]["distance"] = 0
    pq = [(0, orig)]
    while pq:
        _, node = heapq.heappop(pq)
        if node == dest:
            break
        if G.nodes[node]["visited"]:
            continue
        G.nodes[node]["visited"] = True
        for u, v, k in G.out_edges(node, keys=True):
            weight = G.edges[u, v, k]["weight"]
            if G.nodes[v]["distance"] > G.nodes[node]["distance"] + weight:
                G.nodes[v]["distance"] = G.nodes[node]["distance"] + weight
                G.nodes[v]["previous"] = node
                heapq.heappush(pq, (G.nodes[v]["distance"], v))

In [83]:
# Algoritmo A*
def a_star(G, orig, dest):
    def heuristic(node1, node2):
        x1, y1 = G.nodes[node1]["x"], G.nodes[node1]["y"]
        x2, y2 = G.nodes[node2]["x"], G.nodes[node2]["y"]
        return geodesic((y1, x1), (y2, x2)).meters  # Distancia geodésica en metros

    for node in G.nodes:
        G.nodes[node]["g_score"] = float("inf")
        G.nodes[node]["f_score"] = float("inf")
        G.nodes[node]["previous"] = None

    G.nodes[orig]["g_score"] = 0
    G.nodes[orig]["f_score"] = heuristic(orig, dest)
    pq = [(G.nodes[orig]["f_score"], orig)]

    while pq:
        _, node = heapq.heappop(pq)
        if node == dest:
            break
        for u, v, k in G.out_edges(node, keys=True):
            tentative_g_score = G.nodes[node]["g_score"] + G.edges[u, v, k]["weight"]
            if tentative_g_score < G.nodes[v]["g_score"]:
                G.nodes[v]["g_score"] = tentative_g_score
                G.nodes[v]["f_score"] = tentative_g_score + heuristic(v, dest)
                G.nodes[v]["previous"] = node
                heapq.heappush(pq, (G.nodes[v]["f_score"], v))


In [84]:
def get_path_edges_from_previous(G, start_node, end_node):
    path_nodes = []
    curr = end_node
    while curr != start_node:
        prev = G.nodes[curr].get("previous", None)
        if prev is None:
            return None
        path_nodes.append((prev, curr))
        curr = prev
    path_nodes.reverse()
    return path_nodes


def get_path_edges_networkx(G, start_node, end_node):
    real_nodes = nx.shortest_path(G, start_node, end_node, weight="weight")
    real_edges = []
    for u, v in zip(real_nodes[:-1], real_nodes[1:]):
        real_edges.append((u, v))
    return real_edges


def compare_paths(pred_edges, real_edges):
    if pred_edges is None or real_edges is None:
        return {
            "tp": 0, "fp": 0, "fn": 0,
            "precision": 0.0, "recall": 0.0, "f1": 0.0
        }

    # Normalizamos los arcos para no depender de la dirección en la comparación
    norm_pred = {tuple(sorted(e)) for e in pred_edges}
    norm_real = {tuple(sorted(e)) for e in real_edges}

    tp = len(norm_pred & norm_real)
    fp = len(norm_pred - norm_real)
    fn = len(norm_real - norm_pred)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}


def evaluate_pair(G, start_node, end_node, network_type="drive"):
    """
    Evalúa un par (origen, destino) en el grafo:
      - Dijkstra base (sin tráfico)
      - A* base (sin tráfico)
      - Penaliza pesos según tráfico en la ruta A* base (TomTom Traffic Flow)
      - Dijkstra con tráfico (pesos penalizados)
      - A* con tráfico (pesos penalizados)
      - Métricas de tiempo, distancia, precisión/recall para todas las variantes
      - TomTom Routing para ETA global
      - TomTom Traffic Flow sobre la ruta final A* con tráfico para colorear el mapa
    """
    results = {}

    # ===================== 1) DIJKSTRA BASE (sin tráfico) =====================
    t0 = time.perf_counter()
    dijkstra(G, start_node, end_node)
    t1 = time.perf_counter()
    dijkstra_time_base = (t1 - t0) * 1000  # ms

    dijkstra_edges_base = get_path_edges_from_previous(G, start_node, end_node)
    if dijkstra_edges_base:
        dijkstra_dist_base = sum(G.edges[u, v, 0]["length"] for (u, v) in dijkstra_edges_base)
    else:
        dijkstra_dist_base = math.inf

    # ===================== 2) A* BASE (sin tráfico) =====================
    t0 = time.perf_counter()
    a_star(G, start_node, end_node)
    t1 = time.perf_counter()
    astar_time_base = (t1 - t0) * 1000  # ms

    astar_edges_base = get_path_edges_from_previous(G, start_node, end_node)
    if astar_edges_base:
        astar_dist_base = sum(G.edges[u, v, 0]["length"] for (u, v) in astar_edges_base)
    else:
        astar_dist_base = math.inf

    # ===================== 3) Ruta "real" de referencia (networkx) =====================
    real_edges = get_path_edges_networkx(G, start_node, end_node)

    d_base_metrics   = compare_paths(dijkstra_edges_base, real_edges)
    a_base_metrics   = compare_paths(astar_edges_base, real_edges)

    # ===================== 4) Penalizar pesos según tráfico en ruta A* base =====================
    max_factor = 3.0  # tope de penalización (por ejemplo, máximo 3x el peso original)

    if astar_edges_base:
        penalized_edges_info = []

        if astar_edges_base:
            for (u, v) in astar_edges_base:
                try:
                    lat_u, lon_u = G.nodes[u]["y"], G.nodes[u]["x"]
                    flow = tomtom_flow_segment(lat_u, lon_u, zoom=10)

                    curr = flow.get("currentSpeed")
                    free = flow.get("freeFlowSpeed")

                    if curr and free and curr > 0:
                        ratio = curr / free
                        if ratio < 1.0:
                            factor = min(1.0 / ratio, max_factor)
                        else:
                            factor = 1.0
                    else:
                        ratio = None
                        factor = 1.0

                    for k in G[u][v]:

                        # Solo penalizamos si la arista tiene "weight"
                        if "weight" in G[u][v][k]:

                            old_weight = G[u][v][k]["weight"]
                            new_weight = old_weight * factor
                            G[u][v][k]["weight"] = new_weight

                            # Guardar en la lista SOLO si hay weight real
                            penalized_edges_info.append({
                                "u": u,
                                "v": v,
                                "key": k,
                                "old_weight": old_weight,
                                "new_weight": new_weight,
                                "factor": factor,
                                "currentSpeed": curr,
                                "freeFlowSpeed": free,
                                "ratio": ratio,
                            })

                except Exception as e:
                    print(f"[WARN] No se pudo ajustar tráfico en {u}->{v}: {e}")
                    continue

        results["penalized_edges"] = penalized_edges_info


    # ===================== 5) DIJKSTRA CON TRÁFICO (pesos penalizados) =====================
    t0 = time.perf_counter()
    dijkstra(G, start_node, end_node)
    t1 = time.perf_counter()
    dijkstra_time_traffic = (t1 - t0) * 1000  # ms

    dijkstra_edges_traffic = get_path_edges_from_previous(G, start_node, end_node)
    if dijkstra_edges_traffic:
        dijkstra_dist_traffic = sum(G.edges[u, v, 0]["length"] for (u, v) in dijkstra_edges_traffic)
    else:
        dijkstra_dist_traffic = math.inf

    d_traffic_metrics = compare_paths(dijkstra_edges_traffic, real_edges)

    # ===================== 6) A* CON TRÁFICO (pesos penalizados) =====================
    t0 = time.perf_counter()
    a_star(G, start_node, end_node)
    t1 = time.perf_counter()
    astar_time_traffic = (t1 - t0) * 1000  # ms

    astar_edges_traffic = get_path_edges_from_previous(G, start_node, end_node)
    if astar_edges_traffic:
        astar_dist_traffic = sum(G.edges[u, v, 0]["length"] for (u, v) in astar_edges_traffic)
    else:
        astar_dist_traffic = math.inf

    a_traffic_metrics = compare_paths(astar_edges_traffic, real_edges)
    results["astar_edges_base"] = astar_edges_base
    results["astar_edges_traffic"] = astar_edges_traffic


    # ===================== 7) Speedups y eficiencia =====================
    speedup_A_base      = dijkstra_time_base    / astar_time_base    if astar_time_base    > 0 else 0.0
    eficiencia_A_base   = speedup_A_base / 1.0

    speedup_A_traffic   = dijkstra_time_base    / astar_time_traffic if astar_time_traffic > 0 else 0.0
    eficiencia_A_traf   = speedup_A_traffic / 1.0

    speedup_D_traffic   = dijkstra_time_base    / dijkstra_time_traffic if dijkstra_time_traffic > 0 else 0.0
    eficiencia_D_traf   = speedup_D_traffic / 1.0

    # ===================== 8) TomTom ROUTING (ETA global tramo start->end) =====================
    try:
        tt_dist_m, tt_time_s, tt_delay_s = tomtom_from_graph_nodes(G, start_node, end_node)
    except Exception as e:
        print(f"[WARN] Error al llamar a TomTom Routing para {start_node} -> {end_node}: {e}")
        tt_dist_m, tt_time_s, tt_delay_s = None, None, None

    # ===================== 9) TomTom TRAFFIC FLOW sobre la ruta FINAL A* (con tráfico) =====================
    traffic_segments = []
    ratios = []

    if astar_edges_traffic:
        for (u, v) in astar_edges_traffic:
            try:
                lat_u, lon_u = G.nodes[u]["y"], G.nodes[u]["x"]
                flow = tomtom_flow_segment(lat_u, lon_u, zoom=10)

                curr = flow.get("currentSpeed")
                free = flow.get("freeFlowSpeed")
                road_closure = flow.get("roadClosure", False)
                coords_raw = flow.get("coordinates", {}).get("coordinate", [])

                if free and free > 0:
                    ratio = curr / free
                else:
                    ratio = None

                color = traffic_color_from_ratio(ratio, road_closure)
                coords = [(c["latitude"], c["longitude"]) for c in coords_raw]

                traffic_segments.append({
                    "coords": coords,
                    "currentSpeed": curr,
                    "freeFlowSpeed": free,
                    "ratio": ratio,
                    "roadClosure": road_closure,
                    "color": color,
                })

                if ratio is not None:
                    ratios.append(ratio)

            except Exception as e:
                print(f"[WARN] Error en Traffic Flow para arista {u}->{v}: {e}")
                continue

    if ratios:
        avg_ratio = sum(ratios) / len(ratios)
        congested_fraction = sum(1 for r in ratios if r < 0.7) / len(ratios)
    else:
        avg_ratio = None
        congested_fraction = None

    # ===================== 10) Guardar todo en results =====================

    # Dijkstra base
    results["dijkstra_time_ms"] = dijkstra_time_base
    results["dijkstra_dist_m"] = dijkstra_dist_base
    results["dijkstra_precision"] = d_base_metrics["precision"]
    results["dijkstra_recall"] = d_base_metrics["recall"]
    results["dijkstra_f1"] = d_base_metrics["f1"]

    # Dijkstra con tráfico
    results["dijkstra_traffic_time_ms"] = dijkstra_time_traffic
    results["dijkstra_traffic_dist_m"] = dijkstra_dist_traffic
    results["dijkstra_traffic_precision"] = d_traffic_metrics["precision"]
    results["dijkstra_traffic_recall"] = d_traffic_metrics["recall"]
    results["dijkstra_traffic_f1"] = d_traffic_metrics["f1"]
    results["speedup_Dijkstra_traffic_vs_base"] = speedup_D_traffic
    results["eficiencia_Dijkstra_traffic"] = eficiencia_D_traf

    # A* base
    results["astar_time_ms"] = astar_time_base
    results["astar_dist_m"] = astar_dist_base
    results["astar_precision"] = a_base_metrics["precision"]
    results["astar_recall"] = a_base_metrics["recall"]
    results["astar_f1"] = a_base_metrics["f1"]
    results["speedup_A*_vs_Dijkstra"] = speedup_A_base
    results["eficiencia_A*_base"] = eficiencia_A_base

    # A* con tráfico
    results["astar_traffic_time_ms"] = astar_time_traffic
    results["astar_traffic_dist_m"] = astar_dist_traffic
    results["astar_traffic_precision"] = a_traffic_metrics["precision"]
    results["astar_traffic_recall"] = a_traffic_metrics["recall"]
    results["astar_traffic_f1"] = a_traffic_metrics["f1"]
    results["speedup_A*_traffic_vs_Dijkstra_base"] = speedup_A_traffic
    results["eficiencia_A*_traffic"] = eficiencia_A_traf

    # TomTom Routing
    results["tomtom_dist_m"] = tt_dist_m
    results["tomtom_time_s"] = tt_time_s
    results["tomtom_delay_s"] = tt_delay_s

    # Métricas de Traffic Flow (ruta A* final)
    results["traffic_avg_speed_ratio"] = avg_ratio
    results["traffic_congested_fraction"] = congested_fraction
    results["traffic_segments"] = traffic_segments

    
    results["eficiencia"] = results["eficiencia_A*_base"]

    
    return results




def eval_task(args):
    """
    Tarea que se puede correr en paralelo.
    Se hace una copia del grafo para no pisar los 'previous' de otros hilos.
    """
    G, start_node, end_node, start_name, end_name, network_type = args
    G_local = G.copy()  # <- cada proceso/hilo trabaja con su copia

    r = evaluate_pair(G_local, start_node, end_node, network_type=network_type)
    r["origen"] = start_name
    r["destino"] = end_name
    return r


from concurrent.futures import ProcessPoolExecutor

def evaluate_pairs_parallel(G, nodes, point_names, network_type="drive", max_workers=4):
    """
    Versión paralela basada en ProcessPoolExecutor (esta puede ser
    sobrescrita en la celda 17 con la versión que usa ThreadPool en Windows).
    """
    tasks = []
    for i in range(len(point_names) - 1):
        start_name = point_names[i]
        end_name = point_names[i + 1]
        start_node = nodes[start_name]
        end_node = nodes[end_name]
        tasks.append((G, start_node, end_node, start_name, end_name, network_type))

    results = []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        for r in ex.map(eval_task, tasks):
            results.append(r)

    return results



In [85]:
# Mostrar resultados en tabla
def show_results_in_table(total_time, total_distance):
    """
    Muestra los resultados finales en una tabla.
    """
    data = {
        "Algoritmo": ["Dijkstra", "A*"],
        "Distancia Total (km)": [
            total_distance["Dijkstra"] / 1000,  # Convertir de metros a kilómetros
            total_distance["A*"] / 1000
        ],
        "Tiempo Total (minutos)": [
            total_time["Dijkstra"],
            total_time["A*"]
        ]
    }

    # Crear el DataFrame y mostrarlo
    df_results = pd.DataFrame(data)
    display(df_results)


In [86]:
# Obtener las coordenadas de la ruta más corta
def get_route_coordinates(G, orig, dest):
    path = []
    curr = dest
    while curr != orig:
        prev = G.nodes[curr]["previous"]
        if prev is None:
            print("No se pudo encontrar un camino desde el origen al destino.")
            return None
        path.append((G.nodes[curr]["y"], G.nodes[curr]["x"]))
        curr = prev
    path.append((G.nodes[orig]["y"], G.nodes[orig]["x"]))
    path.reverse()
    return path

In [87]:
# Visualizar la ruta con Folium
def plot_route_with_folium(G, orig, dest, route_coordinates):
    folium_map = folium.Map(
        location=[G.nodes[orig]['y'], G.nodes[orig]['x']],
        zoom_start=14,
        tiles='OpenStreetMap'
    )
    folium.Marker(
        location=[G.nodes[orig]['y'], G.nodes[orig]['x']],
        popup="Inicio",
        icon=folium.Icon(color="green", icon="play"),
    ).add_to(folium_map)
    folium.Marker(
        location=[G.nodes[dest]['y'], G.nodes[dest]['x']],
        popup="Destino",
        icon=folium.Icon(color="red", icon="stop"),
    ).add_to(folium_map)
    folium.PolyLine(
        route_coordinates,
        color="blue",
        weight=5,
        opacity=1,
        tooltip="Ruta más corta"
    ).add_to(folium_map)
    return folium_map

In [88]:
# Coordenadas guardadas
saved_coordinates = {
    "UPIIT": (19.323118, -98.233548),
    "Parque de Zacatelco": (19.215691, -98.240524),
    "Soriana Ocotlan": (19.318605, -98.220713),
    "Zoologico del Altiplano": (19.338439, -98.199046),
    "Recinto Ferial": (19.32500854887219, -98.24458295157108),
    "Escalinatas": (19.3163317057408, -98.2432146346621),
    "Jardín botánico": (19.328860233644402, -98.2186988643214),
    "Museo Nacional del Títere": (19.31306644075224, -97.92325948735258),
    "Museo Taurino de Huamantla": (19.315214643452563, -97.92088832789638),
    "Parque de Apizaco": (19.41583668098076, -98.1404040514315),
    "Parque de Tlaxco": (19.614479362946597, -98.11878748334519),
    "Centro Turístico Zacatelco": (19.201482353314457, -98.250818177072),
    "Val'Quirico": (19.19134212101139, -98.28932538116192),
    "Cacaxtla": (19.244591586170426, -98.33845364678213),
}

# Función para obtener coordenadas (seleccionar de lista o ingresar manualmente)
def get_coordinates_with_saved_options():
    print("\n¿Desea seleccionar un punto de la lista predefinida o ingresar coordenadas manualmente?")
    print("  1. Seleccionar de la lista")
    print("  2. Ingresar manualmente")
    while True:
        try:
            option = int(input("Seleccione una opción (1 o 2): ").strip())
            if option == 1:
                print("\nPuntos disponibles:")
                for idx, (name, coord) in enumerate(saved_coordinates.items(), start=1):
                    print(f"  {idx}. {name} - Coordenadas: {coord}")
                while True:
                    try:
                        choice = int(input("Seleccione un punto por su número: ").strip())
                        if 1 <= choice <= len(saved_coordinates):
                            selected_name = list(saved_coordinates.keys())[choice - 1]
                            print(f"Ha seleccionado: {selected_name} - Coordenadas: {saved_coordinates[selected_name]}")
                            return saved_coordinates[selected_name]
                        else:
                            print("Por favor, seleccione un número válido de la lista.")
                    except ValueError:
                        print("Entrada inválida. Por favor, ingrese un número.")
            elif option == 2:
                return get_coordinates()  # Llama a la función manual existente
            else:
                print("Por favor, seleccione una opción válida (1 o 2).")
        except ValueError:
            print("Entrada inválida. Por favor, ingrese un número.")


In [89]:
# Función para agregar puntos dinámicamente
def get_multiple_points():
    points = {}  # Diccionario para almacenar los puntos con sus nombres
    point_counter = 0  # Contador para nombrar los puntos dinámicamente (A, B, C, etc.)

    while True:
        # Generar el nombre del punto (A, B, C, ...)
        point_name = chr(65 + point_counter)  # 65 es el código ASCII de 'A'
        print(f"\nIngrese las coordenadas para el punto {point_name}:")

        # Obtener coordenadas
        coord = get_coordinates_with_saved_options()
        points[point_name] = coord  # Guardar el punto en el diccionario

        # Preguntar si desea agregar otro punto
        while True:
            add_more = input(f"¿Desea agregar otro punto después de {point_name}? (s/n): ").strip().lower()
            if add_more in ["s", "n"]:
                break
            print("Por favor, ingrese 's' para sí o 'n' para no.")

        if add_more == "n":
            break  # Finalizar el bucle si el usuario no desea agregar más puntos

        point_counter += 1  # Incrementar el contador para el próximo punto

    return points


In [90]:
import random

def generate_random_color():
    """
    Genera un color hexadecimal aleatorio.
    """
    return f"#{random.randint(0, 255):02x}{random.randint(0, 255):02x}{random.randint(0, 255):02x}"

def plot_full_route_with_folium(G, nodes, routes, algorithm_name):
    """
    Genera un mapa para un algoritmo específico con puntos y rutas.
    """
    # Crear el mapa centrado en el primer punto
    first_point = list(nodes.values())[0]
    folium_map = folium.Map(
        location=[G.nodes[first_point]['y'], G.nodes[first_point]['x']],
        zoom_start=12,
        tiles='OpenStreetMap'
    )

    # Colores predefinidos para puntos
    predefined_colors = ["blue", "red", "green", "purple", "orange", "pink"]
    color_map = {}

    # Añadir puntos al mapa
    legend_items = []  # Para la leyenda
    for idx, (name, node) in enumerate(nodes.items()):
        if idx < len(predefined_colors):
            color = predefined_colors[idx]
        else:
            # Generar un color aleatorio si se exceden los colores predefinidos
            color = generate_random_color()
        color_map[name] = color  # Guardar el color asignado para la leyenda

        # Añadir el marcador del punto
        folium.Marker(
            location=[G.nodes[node]['y'], G.nodes[node]['x']],
            popup=f"Punto {name}",
            icon=folium.Icon(color=color if idx < len(predefined_colors) else "lightgray", icon="circle"),
        ).add_to(folium_map)

        # Añadir el punto a la leyenda
        legend_items.append(f"<span style='color:{color};'>Punto {name}</span>")

    # Añadir rutas al mapa
    for start_name, end_name, route_coordinates in routes:
        folium.PolyLine(
            route_coordinates,
            color="blue",  # Color de las rutas para el algoritmo
            weight=5,
            opacity=1,
            tooltip=f"Ruta de {start_name} a {end_name} ({algorithm_name})"
        ).add_to(folium_map)

    # Añadir leyenda al mapa
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; background-color: white; padding: 10px; border: 2px solid black;">
    <h4>Leyenda</h4>
    """
    legend_html += "".join(f"<p style='margin:0'>{item}</p>" for item in legend_items)
    legend_html += "</div>"
    folium_map.get_root().html.add_child(folium.Element(legend_html))

    return folium_map



In [ ]:

def agregar_trafico_a_mapa(m, traffic_segments):
    """
    Recibe un mapa Folium ya creado (con rutas Dijkstra/A*)
    y dibuja encima los segmentos de tráfico de TomTom con colores.

    - m: folium.Map (ya con rutas)
    - traffic_segments: lista de dicts como results["traffic_segments"]
    """
    if not traffic_segments:
        # Nada que pintar
        return m

    for seg in traffic_segments:
        coords = seg.get("coords", [])
        color = seg.get("color", "gray")
        if not coords:
            continue

        folium.PolyLine(
            coords,
            color=color,       # verde / amarillo / naranja / rojo / negro
            weight=1,          
            opacity=0.3,
            tooltip=(
                f"Vel. actual: {seg.get('currentSpeed')} km/h | "
                f"Libre: {seg.get('freeFlowSpeed')} km/h | "
                f"ratio: {round(seg.get('ratio', 0), 2) if seg.get('ratio') else 'N/A'}"
            )
        ).add_to(m)

    return m


In [92]:
import os
import pickle

GRAPH_PATH = "tlaxcala_drive.pkl"

def load_tlaxcala_graph(network_type="drive"):
    graph_path = f"tlaxcala_{network_type}.pkl"
    if os.path.exists(graph_path):
        print("Cargando grafo desde disco...")
        with open(graph_path, "rb") as f:
            G = pickle.load(f)
    else:
        print("Descargando el grafo de Tlaxcala...")
        tlaxcala_graph = ox.graph_from_place("Tlaxcala, México", network_type=network_type)
        G = clean_graph(tlaxcala_graph, network_type)
        initialize_edge_styles(G)
        with open(graph_path, "wb") as f:
            pickle.dump(G, f)
        print("Grafo guardado en", graph_path)
    return G


In [93]:
import os
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def eval_task_light(args):
    start_node, end_node, start_name, end_name, network_type = args

    G_local = load_tlaxcala_graph(network_type=network_type)
    r = evaluate_pair(G_local, start_node, end_node, network_type=network_type)
    r["origen"] = start_name
    r["destino"] = end_name
    return r

def evaluate_pairs_parallel(G, nodes, point_names, network_type="drive", max_workers=4):
    tasks = []
    for i in range(len(point_names) - 1):
        sname = point_names[i]
        ename = point_names[i + 1]
        snode = nodes[sname]
        enode = nodes[ename]
        tasks.append((snode, enode, sname, ename, network_type))

    results = []

    Executor = ThreadPoolExecutor if os.name == "nt" else ProcessPoolExecutor

    with Executor(max_workers=max_workers) as ex:
        for r in ex.map(eval_task_light, tasks):
            results.append(r)

    return results


In [94]:
if __name__ == "__main__":
    
    try:
        print("Seleccione los puntos de la ruta:")
        points = get_multiple_points()
        print("\nPuntos ingresados:")
        for name, coord in points.items():
            print(f"  {name}: {coord}")
    
        # Tipo de transporte
        network_type = get_transport_type()
    
        # Cargar/descargar grafo
        G = load_tlaxcala_graph(network_type=network_type)

    
        # Obtener nodos más cercanos
        nodes = {}
        for name, coord in points.items():
            nodes[name] = ox.distance.nearest_nodes(G, coord[1], coord[0])
            print(f"Nodo más cercano al punto {name}: {nodes[name]}")
    
        point_names = list(points.keys())
    
        algorithms = {"Dijkstra": dijkstra, "A*": a_star}
        total_time = {"Dijkstra": 0, "A*": 0}
        total_distance = {"Dijkstra": 0, "A*": 0}
        routes_by_algorithm = {"Dijkstra": [], "A*": []}

        # ====== 1) MÉTRICAS EN PARALELO ======
        metrics_rows = evaluate_pairs_parallel(
            G,
            nodes,
            point_names,
            network_type=network_type,
            max_workers=4
        )
        for r in metrics_rows:
            edges_traf = r.get("astar_edges_traffic", [])
            segments = r.get("traffic_segments", [])
            origen = r.get("origen", "A")
            destino = r.get("destino", "B")

            print(f"Generando mapa para {origen} -> {destino}...")
            mapa_ruta_con_trafico(G, edges_traf, segments, origen, destino)

        # ====== FIN PARALELO ======
    
        # ====== 2) RECORRIDO NORMAL (PARA MAPAS Y TOTALES) ======
        for i in range(len(point_names) - 1):
            start_name = point_names[i]
            end_name = point_names[i + 1]
            print(f"\nCalculando la ruta de {start_name} a {end_name}...")
    
            start_node = nodes[start_name]
            end_node = nodes[end_name]
    
            for algorithm_name, algorithm in algorithms.items():
                print(f"Ejecutando el algoritmo {algorithm_name} de {start_name} a {end_name}...")
                try:
                    algorithm(G, start_node, end_node)
    
                    # reconstruir ruta
                    path = []
                    curr = end_node
                    while curr != start_node:
                        prev = G.nodes[curr]["previous"]
                        path.append((prev, curr))
                        curr = prev
                    path.reverse()
    
                    # distancia y tiempo estimado
                    try:
                        dist = sum(G.edges[u, v, 0]["length"] for u, v in path)
                        avg_speed = 40 if network_type == 'drive' else 5
                        time_est = (dist / 1000) / avg_speed * 60
    
                        total_distance[algorithm_name] += dist
                        total_time[algorithm_name] += time_est
    
                        print(f"{algorithm_name}:")
                        print(f"  Distancia estimada de {start_name} a {end_name}: {dist / 1000:.2f} km")
                        print(f"  Tiempo estimado de {start_name} a {end_name}: {time_est:.2f} minutos")
    
                        # guardar para folium
                        route_coordinates = get_route_coordinates(G, start_node, end_node)
                        routes_by_algorithm[algorithm_name].append(
                            (start_name, end_name, route_coordinates)
                        )
    
                    except Exception as e:
                        print(f"Error al calcular la distancia o tiempo para {algorithm_name}: {e}")
    
                except Exception as e:
                    print(f"Error durante la ejecución de {algorithm_name}: {e}")
        # ====== FIN RECORRIDO NORMAL ======
    
        # ====== 3) MOSTRAR MÉTRICAS ======
        if metrics_rows:
            df_eval = pd.DataFrame(metrics_rows)
            cols = [
                "origen", "destino",
                "dijkstra_time_ms", "astar_time_ms",
                "speedup_A*_vs_Dijkstra", "eficiencia",
                "dijkstra_dist_m", "astar_dist_m",
                "dijkstra_precision", "dijkstra_recall", "dijkstra_f1",
                "astar_precision", "astar_recall", "astar_f1"
            ]
            display(df_eval[cols])
    
            print("\n== Resumen global de métricas ==")
            print("Tiempo medio Dijkstra (ms):", df_eval["dijkstra_time_ms"].mean())
            print("Tiempo medio A* (ms):", df_eval["astar_time_ms"].mean())
            print("Speedup medio A* vs Dijkstra:",
                  (df_eval["dijkstra_time_ms"] / df_eval["astar_time_ms"]).mean())
            print("F1 Dijkstra:", df_eval["dijkstra_f1"].mean())
            print("F1 A*:", df_eval["astar_f1"].mean())

            df_eval.to_csv("resultados_iniciales_tlaxcala.csv", index=False)
        # ====== FIN MÉTRICAS ======
    
        # ====== 4) TOTALES ======
        print("\n--- Totales Finales ---")
        for algorithm_name in algorithms.keys():
            print(f"{algorithm_name}:")
            print(f"  Distancia total estimada: {total_distance[algorithm_name] / 1000:.2f} km")
            print(f"  Tiempo total estimado: {total_time[algorithm_name]:.2f} minutos")
    
        show_results_in_table(total_time, total_distance)
        # ====== FIN TOTALES ======
    
        # ====== 5) MAPAS ======
        for algorithm_name, routes in routes_by_algorithm.items():
            print(f"\nMostrando mapa para el algoritmo {algorithm_name}...")
            
            # Crear el mapa base con Dijkstra o A*
            folium_map = plot_full_route_with_folium(G, nodes, routes, algorithm_name)

            # --- Añadir los colores de tráfico encima del mapa ---
            # metrics_rows contiene traffic_segments de TODOS los pares
            for r in metrics_rows:
                segments = r.get("traffic_segments", [])
                folium_map = agregar_trafico_a_mapa(folium_map, segments)
            # -----------------------------------------------------

            display(folium_map)
        # ====== FIN MAPAS ======


    
    except Exception as e:
        print(f"Error general: {e}")


Seleccione los puntos de la ruta:

Ingrese las coordenadas para el punto A:

¿Desea seleccionar un punto de la lista predefinida o ingresar coordenadas manualmente?
  1. Seleccionar de la lista
  2. Ingresar manualmente


Seleccione una opción (1 o 2):  1



Puntos disponibles:
  1. UPIIT - Coordenadas: (19.323118, -98.233548)
  2. Parque de Zacatelco - Coordenadas: (19.215691, -98.240524)
  3. Soriana Ocotlan - Coordenadas: (19.318605, -98.220713)
  4. Zoologico del Altiplano - Coordenadas: (19.338439, -98.199046)
  5. Recinto Ferial - Coordenadas: (19.32500854887219, -98.24458295157108)
  6. Escalinatas - Coordenadas: (19.3163317057408, -98.2432146346621)
  7. Jardín botánico - Coordenadas: (19.328860233644402, -98.2186988643214)
  8. Museo Nacional del Títere - Coordenadas: (19.31306644075224, -97.92325948735258)
  9. Museo Taurino de Huamantla - Coordenadas: (19.315214643452563, -97.92088832789638)
  10. Parque de Apizaco - Coordenadas: (19.41583668098076, -98.1404040514315)
  11. Parque de Tlaxco - Coordenadas: (19.614479362946597, -98.11878748334519)
  12. Centro Turístico Zacatelco - Coordenadas: (19.201482353314457, -98.250818177072)
  13. Val'Quirico - Coordenadas: (19.19134212101139, -98.28932538116192)
  14. Cacaxtla - Coordena

Seleccione un punto por su número:  1


Ha seleccionado: UPIIT - Coordenadas: (19.323118, -98.233548)


...


== Resumen global de métricas ==
Tiempo medio Dijkstra (ms): 419.786833333319
Tiempo medio A* (ms): 200.84489999999278
Speedup medio A* vs Dijkstra: 2.393790074449464
F1 Dijkstra: 1.0
F1 A*: 0.08477326372063214

--- Totales Finales ---
Dijkstra:
  Distancia total estimada: 32.10 km
  Tiempo total estimado: 48.15 minutos
A*:
  Distancia total estimada: 44.58 km
  Tiempo total estimado: 66.87 minutos


,Algoritmo,Distancia Total (km),Tiempo Total (minutos)
0,Dijkstra,32.096869,48.145304
1,A*,44.581395,66.872093



Mostrando mapa para el algoritmo Dijkstra...



Mostrando mapa para el algoritmo A*...



Mostrando mapa para el algoritmo A*...


In [ ]:
# ===================== GENERACIÓN DE MAPAS DE TRÁFICO =====================

def generar_mapas_trafico(metrics_rows, output_dir="Resultados/Mapas_Trafico"):
    """
    Genera un mapa HTML por cada par origen–destino, usando los segmentos de tráfico
    guardados en results["traffic_segments"].
    """
    os.makedirs(output_dir, exist_ok=True)

    for r in metrics_rows:
        segments = r.get("traffic_segments", [])
        if not segments:
            print(f"[INFO] Sin segmentos de tráfico para {r.get('origen')} -> {r.get('destino')}, se omite mapa.")
            continue

        # Centro del mapa: primer punto del primer segmento
        first_seg = segments[0]
        coords = first_seg.get("coords", [])
        if not coords:
            continue
        center_lat, center_lon = coords[0]

        origen = r.get("origen", "origen")
        destino = r.get("destino", "destino")

        # Limpiar nombres para archivo
        def slugify(name):
            return str(name).replace(" ", "_").replace("/", "-")

        filename = f"mapa_trafico_{slugify(origen)}_{slugify(destino)}.html"
        save_path = os.path.join(output_dir, filename)

        build_traffic_map_from_segments(
            segments,
            center_lat=center_lat,
            center_lon=center_lon,
            save_path=save_path
        )

        print(f"[OK] Mapa de tráfico guardado: {save_path}")


# Si en tu main ya tienes algo como:
# metrics_rows = evaluate_pairs_parallel(...)
# df_metrics = pd.DataFrame(metrics_rows)
# entonces aquí puedes hacer:
try:
    generar_mapas_trafico(metrics_rows)
except NameError:
    print("metrics_rows no está definido en este contexto. Asegúrate de llamarlo después de evaluate_pairs_parallel.")
